In [0]:
df=spark.read.format("parquet").load("s3://pipeline-project-uk/bronze/DimTrack")

In [0]:
display(df)

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import *

In [0]:
df = df.withColumn("track_name",   initcap(trim(col("track_name"))))\
    .withColumn("album_name",   initcap(trim(col("album_name"))))\
    .withColumn("release_date", to_date(col("release_date")))\
    .withColumn("updated_at",   to_timestamp(col("updated_at")))

In [0]:
df.limit(5).display()

In [0]:
df=df.withColumn("duration_min", round(col("duration_sec") / 60, 2))\
    .withColumn("duration_formatted",                                          
                concat_ws(":", 
                    lpad((col("duration_sec") / 60).cast("int").cast("string"), 2, "0"),
                    lpad((col("duration_sec") % 60).cast("string"), 2, "0")))\
    .withColumn("track_length_category",
                when(col("duration_sec") < 120,  lit("Short"))       # < 2 mins
               .when(col("duration_sec") < 240,  lit("Standard"))    # 2–4 mins
               .when(col("duration_sec") < 360,  lit("Long"))        # 4–6 mins
               .otherwise(lit("Extended")))

In [0]:
df.display()

In [0]:
df=df.withColumn("release_year",  year(col("release_date")))\
    .withColumn("release_month", month(col("release_date")))\
    .withColumn("release_quarter", quarter(col("release_date")))

In [0]:
df.display()

In [0]:
from pyspark.sql.functions import datediff, current_date

In [0]:
df=df.withColumn("days_since_last_update", datediff(current_date(), col("updated_at")))


In [0]:
df.display()

In [0]:
df.printSchema()